## 0. Environment check

In [ ]:
import sys, numpy as np, pandas as pd, matplotlib.pyplot as plt, scipy
print("Python", sys.version.split()[0], "| numpy", np.__version__, "| pandas", pd.__version__, "| scipy", scipy.__version__)

try:
    import torch; print("torch", torch.__version__)
except ImportError:
    import tensorflow as tf; print("tensorflow", tf.__version__)

# Representative component test: read one raw test 2 snapshot (the P6 data path)
from pathlib import Path
f = sorted(Path("data/ims/2nd_test").iterdir())[0]
x = pd.read_csv(f, sep=r"\s+", header=None).values
print(f.name, x.shape, "RMS per bearing:", np.sqrt((x**2).mean(axis=0)).round(3))

Python 3.12.16 | numpy 2.5.5 | pandas 3.1.1 | scipy 1.18.1
torch 2.15.0
2004.02.12.10.32.39 (20480, 4) RMS per bearing: [0.074 0.091 0.109 0.054]


## 1. Held-out test 1: inventory only

In [2]:
## 1. Held-out test 1: inventory only (no signal analysis until predictions are committed)
T1 = Path("data/ims/1st_test")
files1 = sorted(p for p in T1.rglob("*") if p.is_file() and p.name[:4].isdigit())
t1 = pd.to_datetime([p.name for p in files1], format="%Y.%m.%d.%H.%M.%S")
print(len(files1), "files |", t1[0], "→", t1[-1])
print("shape of one file:", pd.read_csv(files1[0], sep=r"\s+", header=None).shape)
gaps1 = pd.Series(t1).diff().dt.total_seconds() / 60
print("interval (min): median", gaps1.median(), "| counts:", gaps1.round().value_counts().head(5).to_dict())

2156 files | 2003-10-22 12:06:24 → 2003-11-25 23:39:56
shape of one file: (20480, 8)
interval (min): median 10.0 | counts: {10.0: 1962, 5.0: 148, 9.0: 5, 6.0: 4, 4.0: 4}


### Decisions fixed before looking at test 1 signals

- **Baseline = every snapshot in the first 24 hours of the run**, chosen by time, not by count. Sampling intervals are not constant (at 5-minute spacing, 144 snapshots would cover only about 12 hours), so a fixed count would cover a different stretch of time in each test.
- **One channel per bearing: the first channel of each pair (columns 0, 2, 4, 6).** This rule is chosen without looking at the data, so it can't be tuned to give a good result. Tests 2 and 3 record one channel per bearing, so using both test 1 channels would give the detector a different kind of input on the held-out test.

## 2. Spectral input (shape only), computed from the raw files

In [3]:
from scipy.signal import welch

FS, N_BANDS = 20_000, 128
EDGES = np.linspace(0, FS / 2, N_BANDS + 1)

def spectrum_shape(x):
    """Welch PSD → 128 linear bands → normalised to total power 1 → log10.
    Removing overall power is the P4 lesson: the detector must not rely on loudness alone."""
    f, p = welch(x - x.mean(), fs=FS, nperseg=2048)
    band = np.bincount(np.digitize(f, EDGES[1:-1]), weights=p, minlength=N_BANDS)
    return np.log10(band / band.sum() + 1e-12).astype("float32")

def build_spectra(name, folder, cols):
    out = Path(f"data/derived/spec_{name}.npz")
    if out.exists():
        d = np.load(out, allow_pickle=True)
        return d["S"], d["rms"], pd.to_datetime(d["time"])
    files = sorted(p for p in Path(folder).rglob("*") if p.is_file() and p.name[:4].isdigit())
    S = np.zeros((len(files), 4, N_BANDS), "float32"); R = np.zeros((len(files), 4), "float32")
    for i, p in enumerate(files):
        x = pd.read_csv(p, sep=r"\s+", header=None).values[:, cols]
        for b in range(4):
            S[i, b] = spectrum_shape(x[:, b]); R[i, b] = np.sqrt((x[:, b] ** 2).mean())
        if i % 500 == 0: print(name, i, "/", len(files))
    t = pd.to_datetime([p.name for p in files], format="%Y.%m.%d.%H.%M.%S")
    np.savez_compressed(out, S=S, rms=R, time=t.astype(str).values)
    return S, R, t

S2, R2, t2 = build_spectra("test2", "data/ims/2nd_test", cols=[0, 1, 2, 3])
S3, R3, t3 = build_spectra("test3", "data/ims/4th_test/txt", cols=[0, 1, 2, 3])
print("test 2:", S2.shape, "| test 3:", S3.shape)

test2 0 / 984
test2 500 / 984
test3 0 / 6324
test3 500 / 6324
test3 1000 / 6324
test3 1500 / 6324
test3 2000 / 6324
test3 2500 / 6324
test3 3000 / 6324
test3 3500 / 6324
test3 4000 / 6324
test3 4500 / 6324
test3 5000 / 6324
test3 5500 / 6324
test3 6000 / 6324
test 2: (984, 4, 128) | test 3: (6324, 4, 128)
